In [0]:
%python
from pyspark.sql import functions as F

bronze = spark.table("workspace.bronze.web_events")

# 1. Rejected records: no customer key
rejected = bronze.filter(F.col("custkey").isNull())
rejected.write.mode("overwrite").saveAsTable("workspace.silver.web_events_rejected")

# 2. Clean records
clean = (bronze
    .filter(F.col("custkey").isNotNull())
    .dropDuplicates(["event_id"])
    .withColumn("amount", F.coalesce(F.col("amount").cast("decimal(10,2)"), F.lit(0)))
    .withColumn("event_time", F.to_timestamp(F.regexp_replace("event_time", "/", "-"))))

clean.write.mode("overwrite").saveAsTable("workspace.silver.web_events_clean")

In [0]:
%sql
SELECT * FROM workspace.silver.web_events_clean ORDER BY event_id;

In [0]:
%python
from pyspark.sql import functions as F

orders = spark.table("workspace.bronze.orders")

clean_orders = (orders
    .filter(F.col("o_orderkey").isNotNull() & F.col("o_custkey").isNotNull())
    .dropDuplicates(["o_orderkey"])
    .select("o_orderkey", "o_custkey", "o_orderstatus", "o_totalprice", "o_orderdate"))

clean_orders.write.mode("overwrite").saveAsTable("workspace.silver.orders_clean")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.customer_events AS
SELECT e.event_id, e.custkey, e.event_type, e.amount, e.event_time,
       o.order_count, o.total_order_value
FROM workspace.silver.web_events_clean e
LEFT JOIN (
    SELECT o_custkey,
           COUNT(*) AS order_count,
           SUM(o_totalprice) AS total_order_value
    FROM workspace.silver.orders_clean
    GROUP BY o_custkey
) o    
ON e.custkey = o.o_custkey;

In [0]:
%sql
SELECT * FROM workspace.silver.customer_events ORDER BY event_id;